In [ ]:
import os

# Install Java Development Kit (JDK) 8
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

# Set JAVA_HOME environment variable
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

# Install PySpark and other required libraries
!pip install pyspark==3.5.1
!pip install pandas
!pip install pyngrok==6.0.0 # Older version for compatibility, if needed for Streamlit
!pip install streamlit

print("✅ Environment setup complete: Java, PySpark, pandas, pyngrok, and Streamlit installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 14.3 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488493 sha256=ddafff87c4a33a4617360bd54dea67d99ed4b88f9804c797b62716c4e4a5d81e
  Stored in directory: /root/.cache/pip/wheels/b1/91/5f/283b53010a8016a4ff1c4a1edd99bbe73afacb099645b5471b
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.2
    Uninstalling pyspark-4.0.2:
      Successfully uninstalled pyspark-4.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-conn

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import os
import time

st.set_page_config(layout="wide")
st.title("🚦 Real-Time Traffic Dashboard")

data_path = "/content/output"

# Initial empty dataframe for display
placeholder = st.empty()

while True:
    files = [f for f in os.listdir(data_path) if f.endswith('.csv')]

    df_list = []
    for file in files:
        try:
            df = pd.read_csv(os.path.join(data_path, file))
            df_list.append(df)
        except pd.errors.EmptyDataError:
            # Handle empty CSV files that might be written by Spark Streaming initially
            continue
        except Exception as e:
            st.error(f"Error reading file {file}: {e}")

    if df_list:
        data = pd.concat(df_list, ignore_index=True)
        data['timestamp'] = pd.to_datetime(data['timestamp'], unit='s')
        # Sort by timestamp and drop duplicates to ensure we always have the latest status for each road
        data = data.sort_values(by='timestamp', ascending=False).drop_duplicates(subset=['road', 'timestamp'], keep='first')

        with placeholder.container():
            st.subheader("Live Traffic Data (Latest Updates)")

            # Get unique roads for the multi-select filter
            all_roads = data['road'].unique().tolist()
            selected_roads_multi = st.multiselect("Select Roads for Overall View", all_roads, default=all_roads, key="road_selector_multi")

            # Filter data based on selected roads for overall view
            filtered_data = data[data['road'].isin(selected_roads_multi)]

            st.dataframe(filtered_data.head(10))

            # --- New Feature: Most Congested Road --- Start
            if not filtered_data.empty:
                # Assign numerical values to congestion levels for easier comparison
                congestion_mapping = {'LOW': 0, 'MEDIUM': 1, 'HIGH': 2}
                filtered_data['congestion_numeric'] = filtered_data['congestion'].map(congestion_mapping)

                # Find the road with the highest 'congestion_numeric' in the latest data
                # Group by road and get the maximum numeric congestion encountered
                latest_congestion = filtered_data.groupby('road')['congestion_numeric'].max().reset_index()
                # Sort to find the road with the highest congestion level
                most_congested_road_info = latest_congestion.sort_values(by='congestion_numeric', ascending=False).iloc[0]

                most_congested_road_name = most_congested_road_info['road']
                most_congested_level_numeric = most_congested_road_info['congestion_numeric']
                # Convert numeric congestion back to string for display
                most_congested_level_str = [k for k, v in congestion_mapping.items() if v == most_congested_level_numeric][0]

                st.markdown(f"### 🚨 Most Congested Road: **{most_congested_road_name}** (Level: **{most_congested_level_str}**)")
                st.markdown("---")
            # --- New Feature: Most Congested Road --- End

            st.subheader("Congestion Levels Distribution")
            congestion_counts = filtered_data['congestion'].value_counts().reset_index()
            congestion_counts.columns = ['Congestion Level', 'Count']
            st.bar_chart(congestion_counts.set_index('Congestion Level'))

            st.subheader("Average Vehicle Count per Road")
            avg_vehicles = filtered_data.groupby('road')['vehicle_count'].mean().reset_index()
            avg_vehicles.columns = ['Road', 'Average Vehicle Count']
            st.bar_chart(avg_vehicles.set_index('Road'))

            st.subheader("Average Speed per Road")
            avg_speed = filtered_data.groupby('road')['avg_speed'].mean().reset_index()
            avg_speed.columns = ['Road', 'Average Speed (mph)']
            st.line_chart(avg_speed.set_index('Road'))

            st.subheader("Congestion Trends Over Time")
            # Count occurrences of each congestion level over time
            congestion_time_series = filtered_data.groupby(['timestamp', 'congestion']).size().unstack(fill_value=0)

            # Ensure all congestion levels are present in columns, even if not seen in a time window
            # This makes the chart consistent
            for level in ['LOW', 'MEDIUM', 'HIGH']:
                if level not in congestion_time_series.columns:
                    congestion_time_series[level] = 0

            # Order columns for consistent plotting
            congestion_time_series = congestion_time_series[['LOW', 'MEDIUM', 'HIGH']]
            st.line_chart(congestion_time_series)

            # --- New Feature: Detailed Road Analysis --- Start
            st.markdown("---")
            st.subheader("Detailed Road Analysis")
            selected_road_single = st.selectbox("Select a Road for Detailed View", all_roads, key="road_selector_single")

            if selected_road_single:
                # Filter original 'data' DataFrame for the selected road to show full trends
                detailed_road_data = data[data['road'] == selected_road_single].sort_values(by='timestamp')

                if not detailed_road_data.empty:
                    st.write(f"#### Trends for {selected_road_single}")

                    # Get the min and max timestamps from the detailed data for the slider
                    min_ts = detailed_road_data['timestamp'].min().to_pydatetime()
                    max_ts = detailed_road_data['timestamp'].max().to_pydatetime()

                    # Add a date range slider
                    date_range = st.slider(
                        "Select Time Range",
                        min_value=min_ts,
                        max_value=max_ts,
                        value=(min_ts, max_ts),
                        format="MM/DD/YYYY HH:mm",
                        key="time_range_slider"
                    )

                    # Apply the date range filter
                    filtered_detailed_road_data = detailed_road_data[
                        (detailed_road_data['timestamp'] >= date_range[0]) &
                        (detailed_road_data['timestamp'] <= date_range[1])
                    ]

                    if not filtered_detailed_road_data.empty:
                        st.line_chart(filtered_detailed_road_data.set_index('timestamp')['vehicle_count'].rename("Vehicle Count"))
                        st.line_chart(filtered_detailed_road_data.set_index('timestamp')['avg_speed'].rename("Average Speed (mph)"))

                        # Display congestion level over time for this road
                        # Ensure congestion_mapping is defined here or globally if needed for this section
                        congestion_mapping = {'LOW': 0, 'MEDIUM': 1, 'HIGH': 2}
                        filtered_detailed_road_data['congestion_numeric'] = filtered_detailed_road_data['congestion'].map(congestion_mapping)
                        st.line_chart(filtered_detailed_road_data.set_index('timestamp')['congestion_numeric'].rename("Congestion Level (Numeric)"))
                        st.caption("Congestion Level: 0=LOW, 1=MEDIUM, 2=HIGH")
                    else:
                        st.info(f"No data for {selected_road_single} in the selected time range.")
                else:
                    st.info(f"No data available for {selected_road_single} in the current window.")
            # --- New Feature: Detailed Road Analysis --- End

            # --- New Feature: Map Visualization --- Start
            st.markdown("---")
            st.subheader("Road Locations and Congestion Map")

            # Ensure latitude and longitude columns exist and are numeric
            if 'latitude' in data.columns and 'longitude' in data.columns:
                # Filter to only the latest entry for each road for the map display
                latest_data_for_map = data.drop_duplicates(subset=['road'], keep='first').copy()
                latest_data_for_map = latest_data_for_map[['road', 'latitude', 'longitude', 'congestion']]

                # Convert congestion to a numerical value for potential color coding on map if desired
                congestion_mapping = {'LOW': 1, 'MEDIUM': 2, 'HIGH': 3}
                latest_data_for_map['congestion_level_numeric'] = latest_data_for_map['congestion'].map(congestion_mapping)

                if not latest_data_for_map.empty:
                    st.map(latest_data_for_map,
                           latitude='latitude',
                           longitude='longitude',
                           size='congestion_level_numeric',
                           color=None) # Streamlit map does not directly support color based on column, would need custom plotting library for that
                    st.caption("Map shows latest reported location and congestion level for each road.")
                else:
                    st.info("No geographical data available to display on the map.")
            else:
                st.warning("Latitude or Longitude columns not found in data for map visualization.")
            # --- New Feature: Map Visualization --- End

    else:
        with placeholder.container():
            st.info("Waiting for data...")

    time.sleep(5) # Refresh every 5 seconds


Writing app.py


In [ ]:
from pyngrok import ngrok
import os

# Ensure all ngrok processes are killed before starting a new one
# This is a more forceful termination than ngrok.kill() alone
!killall ngrok > /dev/null 2>&1

# Kill any existing ngrok tunnels started by pyngrok
ngrok.kill()

# Remove existing ngrok configuration files to prevent caching issues
ngrok_config_paths = [
    os.path.expanduser("~/.ngrok2/ngrok.yml"),
    os.path.expanduser("~/.config/ngrok/ngrok.yml")
]
for path in ngrok_config_paths:
    if os.path.exists(path):
        os.remove(path)
        print(f"Removed old ngrok config: {path}")

# Replace with your ngrok authentication token
# You can get one from https://ngrok.com/signup
NGROK_AUTH_TOKEN = "YOUR_AUTH_TOKEN"

# Authenticate ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("✅ ngrok authentication token set.")

# Start a ngrok tunnel for Streamlit
public_url = ngrok.connect(8501)
print(f"✅ ngrok tunnel established: {public_url}")

# Run the Streamlit app in the background
!streamlit run app.py &>/dev/null&

Removed old ngrok config: /root/.ngrok2/ngrok.yml
✅ ngrok authentication token set.
✅ ngrok tunnel established: NgrokTunnel: "https://unperceiving-jordyn-nonallergenic.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
output_path = "/content/output"
checkpoint_path = "/content/checkpoint"

import os
os.makedirs(output_path, exist_ok=True)
os.makedirs(checkpoint_path, exist_ok=True)

query = traffic_df.writeStream \
    .outputMode("append") \
    .format("csv") \
    .option("path", output_path) \
    .option("checkpointLocation", checkpoint_path) \
    .option("header", "true") \
    .trigger(processingTime='5 seconds') \
    .start()

print(f"✅ Spark streaming query started. Writing output to {output_path}")

✅ Generated batch 9 for road Bridge at (33.95, -118.15)
✅ Spark streaming query started. Writing output to /content/output


In [ ]:
from pyspark.sql.functions import when

traffic_df = df.withColumn(
    "congestion",
    when((df.vehicle_count > 70) & (df.avg_speed < 30), "HIGH")
    .when((df.vehicle_count > 40), "MEDIUM")
    .otherwise("LOW")
)

print("✅ Congestion detection logic applied to create traffic_df.")

✅ Congestion detection logic applied to create traffic_df.


In [ ]:
data_path = "/content/traffic_stream"

df = spark.readStream \
    .schema(schema) \
    .option("maxFilesPerTrigger", 1) \
    .json(data_path)

print("✅ Spark streaming DataFrame created to read data from ", data_path)

✅ Generated batch 2 for road Highway-2 at (34.00, -118.50)
✅ Generated batch 3 for road City-Road at (34.03, -118.26)
✅ Spark streaming DataFrame created to read data from  /content/traffic_stream


In [ ]:
import threading
import time
import json
import random
import os

data_path = "/content/traffic_stream"
os.makedirs(data_path, exist_ok=True)  # Ensure the folder exists

roads = ["Highway-1", "Highway-2", "City-Road", "Bridge"]

# Define approximate geographic coordinates for each road
road_locations = {
    "Highway-1": {"latitude": 34.0522, "longitude": -118.2437}, # Los Angeles area
    "Highway-2": {"latitude": 34.0000, "longitude": -118.5000}, # West of LA
    "City-Road": {"latitude": 34.0300, "longitude": -118.2600}, # Downtown LA
    "Bridge": {"latitude": 33.9500, "longitude": -118.1500} # South of LA
}

def generate_data():
    i = 0
    while True:
        try:
            selected_road = random.choice(roads)
            location = road_locations[selected_road]

            data = {
                "road": selected_road,
                "vehicle_count": random.randint(10, 100),
                "avg_speed": random.randint(10, 80),
                "timestamp": time.time(),
                "latitude": location["latitude"],
                "longitude": location["longitude"]
            }

            file_path = f"{data_path}/data_{i}.json"

            with open(file_path, "w") as f:
                json.dump(data, f)

            print(f"✅ Generated batch {i} for road {data['road']} at ({data['latitude']:.2f}, {data['longitude']:.2f})")

            time.sleep(2) # Generate data every 2 seconds
            i += 1

        except Exception as e:
            print("❌ Error in data generator:", e)
            break

# Start data generation in a separate thread
print("Starting data generation thread...")
data_gen_thread = threading.Thread(target=generate_data)
data_gen_thread.daemon = True # Allow the program to exit even if thread is running
data_gen_thread.start()
print("✅ Data generation thread started. Synthetic traffic data will be written to ", data_path)


Starting data generation thread...
✅ Generated batch 0 for road Highway-1 at (34.05, -118.24)
✅ Data generation thread started. Synthetic traffic data will be written to  /content/traffic_stream


In [ ]:
from pyspark.sql.types import *

schema = StructType() \
    .add("road", StringType()) \
    .add("vehicle_count", IntegerType()) \
    .add("avg_speed", IntegerType()) \
    .add("timestamp", DoubleType()) \
    .add("latitude", DoubleType()) \
    .add("longitude", DoubleType())

print("✅ Schema defined successfully.")

✅ Schema defined successfully.


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TrafficProject") \
    .master("local[*]") \
    .getOrCreate()

print("✅ SparkSession initialized successfully.")

✅ SparkSession initialized successfully.


In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import os
import time
import plotly.express as px # Import plotly express

st.set_page_config(layout="wide")
st.title("🚦 Real-Time Traffic Dashboard")

data_path = "/content/output"

# Initialize session state for selections if not present
if 'selected_roads_multi_state' not in st.session_state:
    st.session_state.selected_roads_multi_state = []
if 'selected_road_single_state' not in st.session_state:
    st.session_state.selected_road_single_state = None

# Initial empty dataframe for display
placeholder = st.empty()

while True:
    files = [f for f in os.listdir(data_path) if f.endswith('.csv')]

    df_list = []
    for file in files:
        try:
            df = pd.read_csv(os.path.join(data_path, file))
            df_list.append(df)
        except pd.errors.EmptyDataError:
            # Handle empty CSV files that might be written by Spark Streaming initially
            continue
        except Exception as e:
            st.error(f"Error reading file {file}: {e}")

    if df_list:
        data = pd.concat(df_list, ignore_index=True)
        data['timestamp'] = pd.to_datetime(data['timestamp'], unit='s')
        # Sort by timestamp and drop duplicates to ensure we always have the latest status for each road
        data = data.sort_values(by='timestamp', ascending=False).drop_duplicates(subset=['road', 'timestamp'], keep='first')

        with placeholder.container():
            st.subheader("Live Traffic Data (Latest Updates)")

            # Get unique roads for the multi-select filter
            all_roads = data['road'].unique().tolist()
            if not all_roads: # Handle case where no data/roads are present yet
                st.info("Waiting for data...")
                time.sleep(5)
                continue

            # Generate a dynamic key based on a timestamp to avoid duplicate key errors
            dynamic_key_suffix = str(time.time())

            # --- Multi-select logic for Overall View ---
            # If session state for multi-select is empty or contains roads no longer available, default to all_roads.
            # Otherwise, use the stored session state.
            current_multi_default = st.session_state.selected_roads_multi_state
            if not current_multi_default or not set(current_multi_default).issubset(set(all_roads)):
                current_multi_default = all_roads
            else:
                # Filter out any roads from session_state that are no longer in all_roads (e.g. if data stream changes)
                current_multi_default = [r for r in current_multi_default if r in all_roads]

            selected_roads_multi = st.multiselect(
                "Select Roads for Overall View",
                options=all_roads,
                default=current_multi_default,
                key=f"road_selector_multi_{dynamic_key_suffix}"
            )
            # Update session state with the actual selection from the widget
            st.session_state.selected_roads_multi_state = selected_roads_multi

            # Filter data based on selected roads for overall view
            filtered_data = data[data['road'].isin(selected_roads_multi)]

            st.dataframe(filtered_data.head(10))

            # --- New Feature: Most Congested Road --- Start
            if not filtered_data.empty:
                # Assign numerical values to congestion levels for easier comparison
                congestion_mapping = {'LOW': 0, 'MEDIUM': 1, 'HIGH': 2}
                filtered_data['congestion_numeric'] = filtered_data['congestion'].map(congestion_mapping)

                # Find the road with the highest 'congestion_numeric' in the latest data
                # Group by road and get the maximum numeric congestion encountered
                latest_congestion = filtered_data.groupby('road')['congestion_numeric'].max().reset_index()
                # Sort to find the road with the highest congestion level
                most_congested_road_info = latest_congestion.sort_values(by='congestion_numeric', ascending=False).iloc[0]

                most_congested_road_name = most_congested_road_info['road']
                most_congested_level_numeric = most_congested_road_info['congestion_numeric']
                # Convert numeric congestion back to string for display
                most_congested_level_str = [k for k, v in congestion_mapping.items() if v == most_congested_level_numeric][0]

                st.markdown(f"### 🚨 Most Congested Road: **{most_congested_road_name}** (Level: **{most_congested_level_str}**)")
                st.markdown("---")
            # --- New Feature: Most Congested Road --- End

            st.subheader("Congestion Levels Distribution")
            congestion_counts = filtered_data['congestion'].value_counts().reset_index()
            congestion_counts.columns = ['Congestion Level', 'Count']
            st.bar_chart(congestion_counts.set_index('Congestion Level'))

            st.subheader("Average Vehicle Count per Road")
            avg_vehicles = filtered_data.groupby('road')['vehicle_count'].mean().reset_index()
            avg_vehicles.columns = ['Road', 'Average Vehicle Count']
            st.bar_chart(avg_vehicles.set_index('Road'))

            st.subheader("Average Speed per Road")
            avg_speed = filtered_data.groupby('road')['avg_speed'].mean().reset_index()
            avg_speed.columns = ['Road', 'Average Speed (mph)']
            st.line_chart(avg_speed.set_index('Road'))

            st.subheader("Congestion Trends Over Time")
            # Count occurrences of each congestion level over time
            congestion_time_series = filtered_data.groupby(['timestamp', 'congestion']).size().unstack(fill_value=0)

            # Ensure all congestion levels are present in columns, even if not seen in a time window
            # This makes the chart consistent
            for level in ['LOW', 'MEDIUM', 'HIGH']:
                if level not in congestion_time_series.columns:
                    congestion_time_series[level] = 0

            # Order columns for consistent plotting
            congestion_time_series = congestion_time_series[['LOW', 'MEDIUM', 'HIGH']]
            st.line_chart(congestion_time_series)

            # --- New Feature: Detailed Road Analysis --- Start
            st.markdown("---")
            st.subheader("Detailed Road Analysis")

            # --- Single-select logic for Detailed View ---
            # For st.selectbox, we need to ensure the default value is always in options.
            # If session state value is not set or not in current all_roads, set to the first available road.
            if 'selected_road_single_state' not in st.session_state or \
               st.session_state.selected_road_single_state not in all_roads:
                st.session_state.selected_road_single_state = all_roads[0] if all_roads else None

            # Find the index of the default value for the selectbox
            current_default_single_index = 0
            if st.session_state.selected_road_single_state and st.session_state.selected_road_single_state in all_roads:
                current_default_single_index = all_roads.index(st.session_state.selected_road_single_state)

            selected_road_single = st.selectbox(
                "Select a Road for Detailed View",
                options=all_roads,
                index=current_default_single_index,
                key=f"road_selector_single_{dynamic_key_suffix}" # Dynamic key for slider
            )
            # Update session state with the actual selection from the widget
            st.session_state.selected_road_single_state = selected_road_single

            if selected_road_single:
                # Filter original 'data' DataFrame for the selected road to show full trends
                detailed_road_data = data[data['road'] == selected_road_single].sort_values(by='timestamp')

                if not detailed_road_data.empty:
                    st.write(f"#### Trends for {selected_road_single}")

                    # Get the min and max timestamps from the detailed data for the slider
                    min_ts = detailed_road_data['timestamp'].min().to_pydatetime()
                    max_ts = detailed_road_data['timestamp'].max().to_pydatetime()

                    # Add a date range slider
                    date_range = st.slider(
                        "Select Time Range",
                        min_value=min_ts,
                        max_value=max_ts,
                        value=(min_ts, max_ts),
                        format="MM/DD/YYYY HH:mm",
                        key=f"time_range_slider_{dynamic_key_suffix}"
                    )

                    # Apply the date range filter
                    filtered_detailed_road_data = detailed_road_data[
                        (detailed_road_data['timestamp'] >= date_range[0]) &
                        (detailed_road_data['timestamp'] <= date_range[1])
                    ]

                    if not filtered_detailed_road_data.empty:
                        st.write("##### Vehicle Count Over Time")
                        st.line_chart(filtered_detailed_road_data.set_index('timestamp')['vehicle_count'].rename("Vehicle Count"))
                        st.write("##### Average Speed Over Time")
                        st.line_chart(filtered_detailed_road_data.set_index('timestamp')['avg_speed'].rename("Average Speed (mph)"))

                        # Display congestion level over time for this road
                        # Ensure congestion_mapping is defined here or globally if needed for this section
                        congestion_mapping = {'LOW': 0, 'MEDIUM': 1, 'HIGH': 2}
                        filtered_detailed_road_data['congestion_numeric'] = filtered_detailed_road_data['congestion'].map(congestion_mapping)
                        st.write("##### Congestion Level Over Time")
                        st.line_chart(filtered_detailed_road_data.set_index('timestamp')['congestion_numeric'].rename("Congestion Level (Numeric)"))
                        st.caption("Congestion Level: 0=LOW, 1=MEDIUM, 2=HIGH")
                    else:
                        st.info(f"No data for {selected_road_single} in the selected time range.")
                else:
                    st.info(f"No data available for {selected_road_single} in the current window.")
            # --- New Feature: Detailed Road Analysis --- End

            # --- New Feature: Map Visualization --- Start
            st.markdown("---")
            st.subheader("Road Locations and Congestion Map")

            # Ensure latitude and longitude columns exist and are numeric
            if 'latitude' in data.columns and 'longitude' in data.columns:
                # Filter to only the latest entry for each road for the map display
                latest_data_for_map = data.drop_duplicates(subset=['road'], keep='first').copy()
                # Initial selection should only include columns known to be in `data` at this point
                latest_data_for_map = latest_data_for_map[['road', 'latitude', 'longitude', 'congestion']]

                # Convert congestion to a numerical value for potential color coding on map if desired
                # This creates 'congestion_level_numeric'
                congestion_mapping = {'LOW': 1, 'MEDIUM': 2, 'HIGH': 3}
                latest_data_for_map['congestion_level_numeric'] = latest_data_for_map['congestion'].map(congestion_mapping)

                if not latest_data_for_map.empty:
                    # Define a color map for congestion levels
                    color_map = {'LOW': 'green', 'MEDIUM': 'orange', 'HIGH': 'red'}

                    fig = px.scatter_mapbox(latest_data_for_map,
                                            lat="latitude",
                                            lon="longitude",
                                            color="congestion", # Use the categorical congestion for coloring
                                            color_discrete_map=color_map, # Apply custom color map
                                            size="congestion_level_numeric", # Use numeric for size, e.g., larger for higher congestion
                                            zoom=10, # Adjust zoom level as needed
                                            hover_name="road",
                                            hover_data={"latitude": False, "longitude": False, "congestion_level_numeric": False}, # Corrected to use 'congestion_level_numeric'
                                            mapbox_style="open-street-map", # Use OpenStreetMap style
                                            title="Real-Time Traffic Congestion Map")

                    fig.update_layout(margin={"r":0,"t":50,"l":0,"b":0})
                    st.plotly_chart(fig, use_container_width=True)

                    st.caption("Map shows latest reported location and congestion level for each road.")
                else:
                    st.info("No geographical data available to display on the map.")
            else:
                st.warning("Latitude or Longitude columns not found in data for map visualization.")
            # --- New Feature: Map Visualization --- End

    else:
        with placeholder.container():
            st.info("Waiting for data...")

    time.sleep(5) # Refresh every 5 seconds

Overwriting app.py
